# Software FJ – Partes 1 y 2
**Integrante:** Brayan Stiven Casas Cardenas
**Entrega:** archivo de logs, las 6 excepciones personalizadas, la clase abstracta `Entidad` y la clase `Cliente`.

### ¿Qué hace esta parte, en palabras simples?
- **Log:** es un cuaderno automático donde el programa anota lo que pasa (eventos y errores) en el archivo `software_fj.log`.
- **Excepción personalizada:** es un error con nombre propio, creado por nosotros, para saber exactamente qué falló (por ejemplo, un cliente mal registrado).
- **Clase abstracta (`Entidad`):** es una plantilla general. No se usa directamente; obliga a las clases hijas (como `Cliente`) a tener ciertos métodos.
- **Cliente:** guarda los datos de una persona y **revisa que sean válidos** antes de aceptarlos (encapsulación y validaciones).

Las siguientes partes del grupo (Servicio, servicios especializados, Reserva y simulación) usan estas clases.

## 1. Configuración del archivo de logs

In [1]:
import logging                           # Módulo para escribir mensajes (logs) en un archivo
import re                                # Módulo para revisar que el correo tenga un formato válido
from abc import ABC, abstractmethod      # Herramientas para crear clases abstractas
from datetime import datetime            # Fecha y hora (lo usará la clase Reserva más adelante)

logger = logging.getLogger("SoftwareFJ")        # Crea el "registrador" propio del sistema
logger.setLevel(logging.INFO)                   # Se guardan los mensajes informativos y los errores
logger.handlers.clear()                         # Evita mensajes repetidos si esta celda se ejecuta otra vez

# Se define el archivo donde se guardan los mensajes (mode="w" lo reinicia en cada ejecución)
manejador = logging.FileHandler("software_fj.log", mode="w", encoding="utf-8")
manejador.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))  # Formato: fecha | nivel | mensaje
logger.addHandler(manejador)                    # Se conecta el archivo con el registrador

logger.info("Sistema Software FJ iniciado")     # Primer mensaje que queda anotado en el log

## 2. Excepciones personalizadas (6)

In [2]:
class SoftwareFJError(Exception):               # Excepción base: todas las demás heredan de ella
    """Error general del sistema Software FJ."""


class ClienteInvalidoError(SoftwareFJError):    # Se usa cuando los datos de un cliente no son válidos
    """Datos del cliente inválidos o faltantes."""


class ServicioInvalidoError(SoftwareFJError):   # Se usa cuando los parámetros de un servicio no son válidos
    """Parámetros del servicio inválidos o faltantes."""


class ServicioNoDisponibleError(SoftwareFJError):  # Se usa cuando se pide un servicio que no está disponible
    """El servicio solicitado no está disponible."""


class ReservaError(SoftwareFJError):            # Se usa cuando una reserva no se puede hacer o es una operación no permitida
    """Operación de reserva incorrecta o no permitida."""


class CalculoInconsistenteError(SoftwareFJError):  # Se usa cuando un cálculo de costos da un resultado raro
    """El cálculo de un costo produjo un valor inconsistente."""

## 3. Clase abstracta `Entidad` y clase `Cliente`

In [3]:
class Entidad(ABC):                             # Plantilla general para cualquier "cosa" del sistema
    """Clase abstracta: representa entidades generales del sistema."""

    _contador = 0                               # Contador compartido para darle un número único a cada entidad

    def __init__(self):
        Entidad._contador += 1                  # Suma 1 cada vez que se crea una entidad nueva
        self._id = Entidad._contador            # Guarda ese número como identificador

    @property
    def id(self):                               # Permite leer el identificador (solo lectura)
        return self._id

    @abstractmethod
    def describir(self):                        # Método obligatorio: cada clase hija debe explicar cómo se describe
        """Cada entidad debe describirse a sí misma."""


class Cliente(Entidad):                         # Un cliente es un tipo de entidad
    """Cliente con datos personales protegidos y validados."""

    def __init__(self, nombre, documento, correo, telefono):
        super().__init__()                      # Primero se crea la entidad base (con su id)
        self.nombre = nombre                    # Al asignar cada dato se ejecuta su validación (setter)
        self.documento = documento
        self.correo = correo
        self.telefono = telefono

    @property
    def nombre(self):                           # Leer el nombre
        return self.__nombre                    # Atributo privado: no se toca directamente desde fuera

    @nombre.setter
    def nombre(self, valor):                    # Guardar el nombre, revisando antes que sea válido
        if not isinstance(valor, str) or len(valor.strip()) < 3:     # Debe ser texto de mínimo 3 letras
            raise ClienteInvalidoError("El nombre debe ser texto de al menos 3 caracteres.")
        self.__nombre = valor.strip()           # Si es válido, se guarda sin espacios sobrantes

    @property
    def documento(self):                        # Leer el documento
        return self.__documento

    @documento.setter
    def documento(self, valor):                 # Guardar el documento, revisando que sea válido
        if not str(valor).isdigit() or not 6 <= len(str(valor)) <= 10:   # Solo números, de 6 a 10 dígitos
            raise ClienteInvalidoError("El documento debe ser numérico de 6 a 10 dígitos.")
        self.__documento = str(valor)

    @property
    def correo(self):                           # Leer el correo
        return self.__correo

    @correo.setter
    def correo(self, valor):                    # Guardar el correo, revisando su formato
        # Patrón simple: algo + @ + algo + punto + algo (por ejemplo, ana@correo.com)
        if not isinstance(valor, str) or not re.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+", valor):
            raise ClienteInvalidoError("El correo electrónico no tiene un formato válido.")
        self.__correo = valor

    @property
    def telefono(self):                         # Leer el teléfono
        return self.__telefono

    @telefono.setter
    def telefono(self, valor):                  # Guardar el teléfono, revisando que sea válido
        if not str(valor).isdigit() or len(str(valor)) != 10:            # Solo números y exactamente 10 dígitos
            raise ClienteInvalidoError("El teléfono debe tener exactamente 10 dígitos.")
        self.__telefono = str(valor)

    def describir(self):                        # Implementa el método obligatorio de Entidad
        return f"Cliente #{self.id}: {self.nombre} | Doc: {self.documento} | {self.correo} | Tel: {self.telefono}"

## 4. Prueba rápida de las partes 1 y 2 (se puede borrar al unir todo el proyecto)

In [4]:
# Prueba 1: un cliente válido se crea sin problemas
cliente = Cliente("Laura Gómez", "1012345678", "laura@correo.com", "3001234567")
print(cliente.describir())                      # Muestra los datos del cliente
logger.info("Cliente válido registrado")        # Se anota en el log

# Prueba 2: un cliente con correo inválido genera un error controlado (el programa NO se detiene)
try:
    Cliente("Ana Ruiz", "52123456", "ana-correo", "3201112233")
except ClienteInvalidoError as error:           # Se atrapa nuestra excepción personalizada
    logger.error(f"Registro de cliente fallido: {error}")    # Se anota el error en el log
    print("Error controlado:", error)

# Se muestra el contenido del archivo de logs
with open("software_fj.log", encoding="utf-8") as archivo:
    print(archivo.read())

Cliente #1: Laura Gómez | Doc: 1012345678 | laura@correo.com | Tel: 3001234567
Error controlado: El correo electrónico no tiene un formato válido.
2026-11-15 17:10:32,093 | INFO | Sistema Software FJ iniciado
2026-11-15 17:10:32,111 | INFO | Cliente válido registrado
2026-11-15 17:10:32,111 | ERROR | Registro de cliente fallido: El correo electrónico no tiene un formato válido.

